# Page 9 — "Outcome vs Process Supervision": two graders on the same chains

One set of reasoning traces, graded twice. An **outcome** grader looks only at the final answer.
A **process** grader looks at every step. This notebook measures what the second one buys.

Three measurements, all from one generation pass:

1. **Detection** — among traces that reached the *right* answer, how many contain a step that is
   demonstrably false. Every one of those is a case outcome grading signs off on and rewards.
2. **Localisation** — among traces that reached the *wrong* answer, how much of the chain was
   fine up to the break. Outcome grading condemns all of it; process grading condemns from the
   break onward.
3. **Selection** — a process-aware rung on page 8's selector ladder, if page 8's dump is present.

**Two graders, and one of them has to earn its labels.** The arithmetic checker is deterministic:
where it judges, it is right by construction, and a reader can check any label by hand. The
process reward model is a model, so its labels are opinions until measured. Section 6 measures
them against the checker on the steps both can see, and reports a reliability rather than
claiming one. That is the same move page 8 makes with its verifiers.

All rationale lives in [`docs/page_9_refactor.md`](../docs/page_9_refactor.md).


## How to run this

| Mode | Generator | Process grader | Items | Where it finishes |
|---|---|---|---|---|
| `QUICK` | Qwen2.5-Math-1.5B-Instruct | checker only | 24 | a laptop CPU, ~30 min |
| `FULL` | Qwen2.5-Math-7B-Instruct | checker + Qwen2.5-Math-PRM-7B | 160 | a 16 GB GPU, ~2 h |

`QUICK` produces every measurement from the deterministic checker alone. The PRM is an
enhancement, not a dependency: if it is off or fails to load, sections 3a and 3b still run and
the page still has its numbers. That ordering is deliberate, because the checker's labels are
the ones the page can call facts.

**Why Qwen2.5 rather than pages 5 and 6's Qwen3.** The only open process reward model in reach
is `Qwen2.5-Math-PRM-7B`, and a PRM is most trustworthy on the family it was trained around, so
the generator is matched to it. Page 9's subject is the *grader*, not the generator, which is
why deviating from Act 2's instrument costs this page nothing. The manifest records both.

**Generator and PRM are never resident together** — 7B of each will not fit 16 GB. The generator
is swept, then freed, then the PRM is loaded.

```bash
pip install -r requirements.txt
```


In [ ]:
# ---------------------------------------------------------------------------
# The knobs. Everything else is derived from these.
# ---------------------------------------------------------------------------
RUN_MODE = "QUICK"        # "QUICK" (laptop CPU) or "FULL" (16 GB GPU)
SEED = 123                # selects items; generation is greedy and unseeded

PRESETS = {
    "QUICK": dict(
        gen_model="Qwen/Qwen2.5-Math-1.5B-Instruct",
        n_items=24,
        max_new_tokens=512,
        use_prm=False,                 # the checker alone carries QUICK
    ),
    "FULL": dict(
        gen_model="Qwen/Qwen2.5-Math-7B-Instruct",
        n_items=160,
        max_new_tokens=1024,
        use_prm=True,
    ),
}
CFG = PRESETS[RUN_MODE]

PRM_MODEL = "Qwen/Qwen2.5-Math-PRM-7B"
USE_PRM = CFG["use_prm"]
MAX_NEW_TOKENS = CFG["max_new_tokens"]

# Traces run to their natural length. Page 6 caps generation on purpose; here a cap
# that binds would be a defect, because a chain we cut off is broken by US and would
# then be counted as the model reasoning badly. The integrity cell reports the rate.

print(f"mode={RUN_MODE}  generator={CFG['gen_model']}")
print(f"process grader: arithmetic checker" + (f" + {PRM_MODEL}" if USE_PRM else " only"))
print(f"items={CFG['n_items']}  cap={MAX_NEW_TOKENS} (should not bind)")


In [ ]:
import gc
import hashlib
import json
import random
import re
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

import torch
from huggingface_hub import snapshot_download
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# The inherited harness: text-in, text-out, no model dependency.
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
    load_math500_test,
    render_prompt,
    extract_final_candidate,
    grade_answer,
)

OUT_DIR = Path("out/page_09")
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

TRACES_PATH = OUT_DIR / "traces.jsonl"
DERIVED_PATH = OUT_DIR / "derived.json"
MANIFEST_PATH = OUT_DIR / "manifest.json"

mean = lambda xs: sum(xs) / max(len(xs), 1)

print("torch", torch.__version__)


## 1. The outcome grader

This is the grader the whole page is arguing about, and we inherit it unchanged from ch03:
`extract_final_candidate` pulls the last `\boxed{...}` out of a completion, `grade_answer`
checks equivalence with LaTeX canonicalisation and SymPy.

Note what it can and cannot see. It reads the last line. It has no opinion about the hundred
tokens before it, which is the entire point of this page.


In [ ]:
CHECKS = [
    # (raw completion, ground truth, expected)
    (r"The answer is \boxed{9}.",                       "9",    True),
    (r"27",                                              "9",    False),
    (r"\boxed{\frac{1}{2}}",                            "0.5",  True),   # LaTeX vs decimal
    (r"\boxed{1,024}",                                  "1024", True),   # thousands separator
    # right answer via nonsense: the outcome grader says yes, and that is the page's subject
    (r"Cancel the 6s: 16/64 -> 1/4. So \boxed{\frac{1}{4}}", "0.25", True),
    (r"I am not sure.",                                  "9",    False),
]

rows = [(raw, gt, exp, bool(grade_answer(extract_final_candidate(raw), gt)))
        for raw, gt, exp in CHECKS]
failures = [r for r in rows if r[3] != r[2]]

if failures:
    for raw, gt, exp, got in rows:
        print(f"{'ok ' if got == exp else 'FAIL'} {extract_final_candidate(raw)!r:<28} "
              f"vs {gt!r:<10} -> {got}")
    raise AssertionError(f"{len(failures)} grader check(s) behaved unexpectedly")

print(f"outcome grader preflight: {len(rows)}/{len(rows)} as expected")
print("note the fifth case: a right answer reached by nonsense grades as correct. That is not")
print("a bug in the grader -- it is the thing this page exists to measure.")


## 2. The item set

MATH-500, levels 4 and 5. Two reasons, and only the first is shared with pages 5 and 6.

**Headroom**, as everywhere in Act 2: a set the model aces produces no wrong answers, and
section 3b needs wrong answers to have anything to localise in.

**Chain length.** Step grading needs chains with steps in them. Easy problems get one-line
solutions, and a one-step chain cannot show a break partway through.


In [ ]:
math500 = load_math500_test(local_path=DATA_DIR / "math500_test.json")

band = sorted((r for r in math500 if int(r["level"]) in (4, 5)),
              key=lambda r: r["unique_id"])          # sort before shuffling
random.Random(SEED).shuffle(band)
items = band[: CFG["n_items"]]
assert len(items) == CFG["n_items"], "not enough problems at levels 4-5"


def sha256_of(strings):
    h = hashlib.sha256()
    for s in strings:
        h.update(s.encode("utf-8"))
        h.update(b"\n")
    return h.hexdigest()


ITEM_SET_HASH = sha256_of(r["unique_id"] for r in items)

from collections import Counter
lv = "+".join(f"L{n}x{c}" for n, c in sorted(Counter(int(r["level"]) for r in items).items()))
print(f"{len(items)} items of {len(band)} available at levels 4-5   {lv}   "
      f"sha256 {ITEM_SET_HASH[:12]}")


## 3. Why this page generates its own traces

It would be cheaper to score page 6's dump. It would also be wrong, and this is the one design
decision on this page most worth understanding.

**Page 6's traces are cut off and forced to commit.** That is its experiment: cap the model,
inject `</think>`, hand it `\boxed{` and make it answer. A chain broken that way is broken by
*us*. Scoring it for bad reasoning would measure our own intervention and report it as the
model's error, which would corrupt every number on this page.

The same objection applies to page 7's high-N samples, which are budget-starved by construction.

So page 9 runs its own pass: **natural length, no cap that binds, no forcing, greedy.** What
the model writes is what it chose to write, and any broken step in it is the model's.

Page 8's dump is still useful, because section 3c reads *selections* rather than traces, and a
selector ladder does not care how the candidates were produced. That section is optional and
skips itself if the dump is absent.


In [ ]:
device = get_device()
DTYPE = torch.float32 if device.type == "cpu" else torch.bfloat16

WARMUP_TOK_S = None


def load_causal(model_id):
    """Four timed stages. from_pretrained draws no bar for single-shard checkpoints,
    so without this the cell sits blank and you cannot tell fetching from loading."""
    global WARMUP_TOK_S
    print(f"[1/4] fetch      {model_id}")
    t0 = time.time()
    local = snapshot_download(model_id)              # draws its own per-file bars
    print(f"      {time.time() - t0:>6.1f}s  (cached after the first run)")

    print("[2/4] load       tokenizer and weights")
    t0 = time.time()
    tok = AutoTokenizer.from_pretrained(local)
    mdl = AutoModelForCausalLM.from_pretrained(local, dtype=DTYPE).to(device).eval()
    got = next(mdl.parameters()).dtype
    if got != DTYPE:
        raise RuntimeError(
            f"asked for {DTYPE}, weights loaded as {got}. Almost certainly transformers "
            f"< 4.56, where the argument is spelled torch_dtype= and dtype= is ignored.")
    print(f"      {time.time() - t0:>6.1f}s")

    print("[3/4] warm up    16 tokens")
    t0 = time.time()
    with torch.inference_mode():
        mdl.generate(**tok("Hello", return_tensors="pt").to(device),
                     max_new_tokens=16, do_sample=False,
                     pad_token_id=tok.pad_token_id if tok.pad_token_id is not None
                     else tok.eos_token_id)
    dt = time.time() - t0
    WARMUP_TOK_S = 16 / max(dt, 1e-6)
    print(f"      {dt:>6.1f}s  ~{WARMUP_TOK_S:.1f} tokens/s")

    n = sum(p.numel() for p in mdl.parameters())
    print(f"[4/4] ready      {n/1e9:.2f}B params  {got}  on {device}\n")
    return mdl, tok, n


def free_memory(*objs):
    """Deletion has to happen at the call site: handing a model to a function that
    dels its own parameter frees nothing, because the caller still holds a reference."""
    del objs
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    mps = getattr(torch.backends, "mps", None)
    if mps is not None and mps.is_available():
        torch.mps.empty_cache()


model, tokenizer, N_PARAMS = load_causal(CFG["gen_model"])


## 4. The generation pass

One chain per problem, greedy, run to natural length. The prompt is ch03's `render_prompt`
plus a step-by-step instruction, because the process grader needs a chain with visible steps
and a model that blurts gives it nothing to grade.

`tokens` is recorded as token strings so the page can reveal a trace one real token at a time,
the same contract pages 5 and 6 use. End-of-turn tokens are stripped: they are how the model
signals it has finished, not something it wrote.


In [ ]:
COT_SUFFIX = "\n\nSolve this step by step, one step per line."
RUN_ID = f"{RUN_MODE.lower()}-{CFG['gen_model'].split('/')[-1]}-{ITEM_SET_HASH[:8]}"


def _id_of(t):
    i = tokenizer.convert_tokens_to_ids(t)
    return i if isinstance(i, int) and i >= 0 and tokenizer.convert_ids_to_tokens(i) == t else None


EOS_IDS = {i for i in (tokenizer.eos_token_id, _id_of("<|im_end|>"), _id_of("<|endoftext|>"))
           if i is not None}


def incremental_token_strings(ids, tok):
    """One string per token such that "".join(pieces) == tok.decode(ids). Decoding
    tokens singly is unsafe: byte-level BPE can split one character across two."""
    pieces, prev = [], ""
    for k in range(1, len(ids) + 1):
        cur = tok.decode(ids[:k], skip_special_tokens=True)
        pieces.append(cur[len(prev):])
        prev = cur
    return pieces


@torch.inference_mode()
def generate(question):
    chat = tokenizer.apply_chat_template(
        [{"role": "user", "content": render_prompt(question) + COT_SUFFIX}],
        tokenize=False, add_generation_prompt=True)
    enc = tokenizer(chat, return_tensors="pt", add_special_tokens=False).to(device)

    out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS,
                         do_sample=False, temperature=None, top_p=None, top_k=None,
                         pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id
                         is not None else tokenizer.eos_token_id)
    new_ids = out[0][enc["input_ids"].shape[1]:].tolist()

    truncated = len(new_ids) >= MAX_NEW_TOKENS     # decided before stripping, on the raw count
    while new_ids and new_ids[-1] in EOS_IDS:
        new_ids.pop()

    text = tokenizer.decode(new_ids, skip_special_tokens=True)
    toks = incremental_token_strings(new_ids, tokenizer)
    assert "".join(toks) == text, "token strings do not reconstruct the decoded text"
    return dict(trace=text, tokens=toks, tokens_generated=len(new_ids), truncated=truncated)


def load_done():
    done = {}
    if TRACES_PATH.exists():
        for line in TRACES_PATH.open(encoding="utf-8"):
            if line.strip():
                r = json.loads(line)
                if r.get("run_id") == RUN_ID:
                    done[r["problem_id"]] = r
    return done


DONE = load_done()
print(f"run_id={RUN_ID}   {len(DONE)} traces already on disk")

records = []
bar = tqdm(items, desc="generating", unit="item")
for row in bar:
    if row["unique_id"] in DONE:
        records.append(DONE[row["unique_id"]])
        continue
    g = generate(row["problem"])
    extracted = extract_final_candidate(g["trace"])
    rec = {
        "run_id": RUN_ID, "problem_id": row["unique_id"], "level": int(row["level"]),
        "subject": row["subject"], "question": row["problem"], "ground_truth": row["answer"],
        "trace": g["trace"], "tokens": g["tokens"],
        "tokens_generated": g["tokens_generated"], "truncated": g["truncated"],
        "extracted_answer": extracted,
        "outcome_correct": bool(grade_answer(extracted, row["answer"])),
    }
    with TRACES_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    DONE[rec["problem_id"]] = rec
    records.append(rec)
    bar.set_postfix(acc=f"{mean([r['outcome_correct'] for r in records]):.2f}")

n_right = sum(r["outcome_correct"] for r in records)
trunc = sum(r["truncated"] for r in records)
print(f"\n{len(records)} traces: {n_right} right, {len(records) - n_right} wrong "
      f"({n_right/len(records):.0%} outcome accuracy)")
print(f"cap bound on {trunc} of them ({trunc/len(records):.1%}) -- above a few percent, raise")
print("MAX_NEW_TOKENS: a chain WE cut is not a chain the model got wrong.")


## 5. Steps, and the arithmetic checker

Process grading needs to know where a step ends. The splitting rule is the one page 5 uses, so
a trace segmented here and a trace segmented there break the same way: blank-line paragraphs,
then newlines, then sentences, taking the first rule that actually divides the trace.

### The precision rule, which is the whole design

The checker may judge **only claims that are closed-form numeric on both sides, with no
symbols.** `3/4 = 9/12` is checkable. `Let x = 5` is a definition, not a false claim. `f(2) = 7`
depends on a definition made earlier in the chain. A checker that flagged those as false would
be worse than no checker at all, because the single thing this one has going for it is that its
labels are facts a reader can verify by hand.

So every step gets one of three labels, and the third is not a hole to hide:

| label | meaning |
|---|---|
| `true` | every numeric claim in the step checks out |
| `false` | at least one numeric claim is arithmetically wrong |
| `unknown` | nothing in the step is a closed-form numeric claim |

High precision, low recall, stated openly. The preflight below includes the page's own canonical
example, cancelling the sixes in 16/64, precisely because the checker gets it **wrong** in the
generous direction: the arithmetic is true and the method is nonsense. That is the limit a
deterministic checker cannot cross, and it is the reason section 6 brings in a model.


In [ ]:
from sympy import simplify, sympify

MIN_STEP_CHARS = 12          # same rule as page 5; a digit, not length, marks a real step


def _raw_spans(text):
    for pattern in (r"\n\s*\n", r"\n", r"(?<=[.!?])\s+"):
        spans, pos = [], 0
        for m in re.finditer(pattern, text):
            if m.start() > pos:
                spans.append((pos, m.start()))
            pos = m.end()
        if pos < len(text):
            spans.append((pos, len(text)))
        spans = [(a, b) for a, b in spans if text[a:b].strip()]
        if len(spans) > 1:
            return spans
    return [(0, len(text))] if text.strip() else []


def segment_steps(text, tokens):
    assert "".join(tokens) == text, "token strings do not reconstruct the trace"
    raw = _raw_spans(text)
    spans = []
    for k, (a, b) in enumerate(raw):
        piece = text[a:b].strip()
        mergeable = (len(piece) < MIN_STEP_CHARS and not re.search(r"\d", piece)
                     and k != len(raw) - 1 and "\\boxed" not in piece)
        if spans and mergeable:
            spans[-1] = (spans[-1][0], b)
        else:
            spans.append((a, b))

    ends, run = [], 0
    for t in tokens:
        run += len(t)
        ends.append(run)

    def token_at(i):
        lo, hi = 0, len(ends)
        while lo < hi:
            mid = (lo + hi) // 2
            lo, hi = (mid + 1, hi) if ends[mid] <= i else (lo, mid)
        return lo

    steps = [{"text": text[a:b].strip(), "token_start": token_at(a)} for a, b in spans]
    for j, s in enumerate(steps):
        s["token_end"] = steps[j + 1]["token_start"] if j + 1 < len(steps) else len(tokens)
    return steps


def _to_sympy_text(s):
    """LaTeX-ish maths to something sympify can read. Deliberately narrow."""
    s = re.sub(r"\\left|\\right", "", s)
    s = re.sub(r"\\[dt]?frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}", r"((\1)/(\2))", s)
    s = (s.replace("\\times", "*").replace("\\cdot", "*").replace("\\div", "/")
          .replace("^", "**").replace("$", "").replace("\\", ""))
    s = re.sub(r"(?<=\d),(?=\d\d\d\b)", "", s)         # thousands separators only
    return re.sub(r"\s+", "", s)


def numeric_value(text):
    """A sympy number if the text is closed-form numeric, else None.

    Any letter disqualifies it. That rejects 'x', 'f(2)', 'pi' and 'sqrt' alike, which
    costs recall and buys the only thing that matters here: no false positives."""
    t = _to_sympy_text(text)
    if not t or not re.search(r"\d", t) or re.search(r"[A-Za-z]", t):
        return None
    try:
        e = sympify(t, rational=True)
    except Exception:
        return None
    return e if getattr(e, "is_number", False) and not e.free_symbols else None


def _strip_prose(side):
    """Drop whole words of prose from the ends of one side of an equation.

    'So 8 * 7' -> '8 * 7' and '56 apples' -> '56', because a claim does not stop being
    checkable for having a sentence wrapped round it. Three guards keep this from
    inventing claims that were never made:

    * only tokens of TWO OR MORE letters go. Single letters are variables, and
      stripping the 'n' from 'n + 1' would leave '+ 1', which parses to 1 and would
      make 'Let n = 4, so n + 1 = 5' read as the false claim 1 = 5.
    * 'f(2)' is one token and is not purely alphabetic, so it survives, meets the
      no-letters rule and is skipped -- rather than being mined for a '(2)'.
    * if stripping leaves the side starting on an operator we amputated an operand,
      so refuse to judge it at all.
    """
    toks = side.split()
    stripped = False
    while toks and (toks[0].endswith(":")                       # a step label, 'Step 3:'
                    or (toks[0].isalpha() and len(toks[0]) > 1)):
        toks.pop(0)
        stripped = True
    while toks and toks[-1].isalpha() and len(toks[-1]) > 1:
        toks.pop()
    out = " ".join(toks).rstrip(".")
    if stripped and re.match(r"^[+*/^]|^-\s", out):
        return ""
    return out


def check_step(text):
    """-> (label, judged_claim). A step is false if ANY numeric claim in it is."""
    # Thousands separators first: splitting on commas below would otherwise cut
    # '1,024' in half and turn a valid claim into an unparseable one.
    text = re.sub(r"(?<=\d),(?=\d\d\d\b)", "", text)

    judged = None
    for chunk in re.split(r"[,;]|\bso\b|\bSo\b|\bthus\b|\band\b|\bTherefore\b", text):
        if "=" not in chunk:
            continue
        vals = [numeric_value(_strip_prose(p)) for p in chunk.split("=")]
        if len(vals) < 2 or any(v is None for v in vals):
            continue
        judged = chunk.strip()
        if any(simplify(vals[0] - v) != 0 for v in vals[1:]):
            return "false", judged
    return ("true", judged) if judged else ("unknown", None)


PREFLIGHT = [
    # --- claims it should judge ------------------------------------------------
    ("3/4 = 9/12",                       "true"),
    ("17/12 - 5/12 = 1",                 "true"),
    ("2 + 2 = 5",                        "false"),
    ("So 8 * 7 = 54.",                   "false"),   # prose in front of the claim
    ("Step 3: 100 - 37 = 62",            "false"),   # a step label in front of it
    ("We get 8 * 7 = 56 apples",         "true"),    # units after it
    ("1,024 = 1024",                     "true"),    # thousands separator, not a chunk break
    (r"\frac{3}{4} = \frac{9}{12}",      "true"),
    (r"$5 \times 4 = 20$",               "true"),
    # --- claims it must refuse: every one of these is a false positive waiting --
    ("Let x = 5",                        "unknown"),  # a definition, not a claim
    ("f(2) = 7",                         "unknown"),  # depends on an earlier definition
    ("Let n = 4, so n + 1 = 5",          "unknown"),  # dropping 'n' would leave '+1 = 5'
    ("Substituting, 2x = 10",            "unknown"),  # has an unknown in it
    ("x = 3 and y = 4",                  "unknown"),
    (r"\sqrt{16} = 4",                   "unknown"),  # true, but not closed-form numeric here
    ("(3, 5) = (3, 5)",                  "unknown"),  # a tuple, not arithmetic
    ("The common denominator is 12",     "unknown"),  # no claim at all
    ("The area is 12 square units",      "unknown"),
    # --- the blind spot, stated as a test rather than as a caveat --------------
    ("Cancel the 6s, so 16/64 = 1/4",    "true"),     # arithmetic true, method nonsense
]
for text, expect in PREFLIGHT:
    got, claim = check_step(text)
    flag = "ok " if got == expect else "FAIL"
    print(f"{flag} {text!r:<36} -> {got:<8} {('claim ' + repr(claim)) if claim else ''}")
    assert got == expect, f"checker misjudged {text!r}: expected {expect}, got {got}"

print("\nthe last case is the one to notice: the arithmetic is true, the method is nonsense,")
print("and a deterministic checker cannot tell. That gap is what section 6 is for.")


In [ ]:
# Segment and check every trace.
for r in tqdm(records, desc="checking steps", unit="trace"):
    steps = segment_steps(r["trace"], r["tokens"])
    for s in steps:
        s["check"], s["claim"] = check_step(s["text"])
    r["steps"] = steps
    r["n_steps"] = len(steps)
    r["first_false"] = next((i for i, s in enumerate(steps) if s["check"] == "false"), None)
    r["n_false"] = sum(s["check"] == "false" for s in steps)
    r["n_checkable"] = sum(s["check"] in ("true", "false") for s in steps)

all_steps = [s for r in records for s in r["steps"]]
lab = Counter(s["check"] for s in all_steps)
print(f"{len(all_steps)} steps across {len(records)} traces")
for k in ("true", "false", "unknown"):
    print(f"  {k:<8} {lab[k]:>5}  {lab[k]/len(all_steps):>5.1%}")
print(f"\nmedian steps per trace: {sorted(r['n_steps'] for r in records)[len(records)//2]}")
print(f"traces with at least one checkable step: "
      f"{sum(r['n_checkable'] > 0 for r in records)}/{len(records)}")


## 6. The process reward model, and measuring whether to believe it

The checker cannot see method errors. A model can, and `Qwen2.5-Math-PRM-7B` is trained to emit
a score per reasoning step, which is process supervision in the literal sense.

But a PRM's labels are opinions. Putting them on the page unexamined would repeat the defect
this refactor exists to remove — the current page reveals hand-written booleans and calls them a
PRM's verdict. So the PRM does not get to be the authority here. It gets **measured against the
checker on the steps both can see**, which is exactly page 8's move: verifier reliability as a
measured property rather than a claimed one.

The checker is the right yardstick for this because on its own subset it is right by
construction. Where they disagree, the checker is correct and the PRM is wrong.

Two things follow. If agreement is high, the PRM's judgements on the *unknown* steps (the ones
the checker cannot reach) are worth reporting, with the measured reliability attached. If
agreement is poor, the page says so and leans on the checker alone. Either outcome is a result.

The generator is freed first: two 7B models will not share 16 GB.


In [ ]:
del model, tokenizer          # the caller's references are what keep it alive
free_memory()

PRM_READY = False
prm = prm_tok = None

if USE_PRM:
    try:
        print(f"[1/3] fetch      {PRM_MODEL}")
        t0 = time.time()
        prm_local = snapshot_download(PRM_MODEL)
        print(f"      {time.time() - t0:>6.1f}s")

        print("[2/3] load       PRM weights")
        t0 = time.time()
        from transformers import AutoModel
        prm_tok = AutoTokenizer.from_pretrained(prm_local, trust_remote_code=True)
        prm = AutoModel.from_pretrained(prm_local, dtype=DTYPE,
                                        trust_remote_code=True).to(device).eval()
        print(f"      {time.time() - t0:>6.1f}s")
        PRM_READY = True
    except Exception as e:
        print(f"\nPRM unavailable ({type(e).__name__}: {e}).")
        print("Continuing with the deterministic checker alone -- sections 3a and 3b still")
        print("produce their numbers, and derived.json records that the PRM did not run.")
else:
    print(f"USE_PRM is off in {RUN_MODE}. The checker carries every measurement below;")
    print("run FULL to add the model-based grader and its reliability measurement.")


In [ ]:
STEP_SEP = "<extra_0>"          # the token this PRM scores at, one per step


@torch.inference_mode()
def prm_scores(question, step_texts):
    """One score in [0,1] per step. Higher means the PRM thinks the step is sound."""
    body = STEP_SEP.join(t.strip() for t in step_texts) + STEP_SEP
    chat = prm_tok.apply_chat_template(
        [{"role": "system", "content": "Please reason step by step."},
         {"role": "user", "content": question},
         {"role": "assistant", "content": body}],
        tokenize=False, add_generation_prompt=False)
    ids = prm_tok(chat, return_tensors="pt").to(device)
    out = prm(**ids)
    logits = out[0] if isinstance(out, (tuple, list)) else getattr(out, "logits", out)
    sep_id = prm_tok.encode(STEP_SEP, add_special_tokens=False)[0]
    mask = ids["input_ids"] == sep_id
    probs = torch.softmax(logits.float(), dim=-1)[mask]      # one row per separator
    return probs[:, -1].tolist()                             # P(step is good)


if PRM_READY:
    # Preflight: a clean chain against the same chain with one step corrupted. If the
    # PRM cannot separate those, its scores are noise and nothing below should use them.
    q = "What is 3/4 + 2/3?"
    clean = ["The common denominator is 12.", "3/4 = 9/12 and 2/3 = 8/12.",
             "9/12 + 8/12 = 17/12.", "So the answer is \\boxed{17/12}."]
    dirty = list(clean)
    dirty[2] = "9/12 + 8/12 = 15/12."                        # step 3 is now false

    s_clean, s_dirty = prm_scores(q, clean), prm_scores(q, dirty)
    assert len(s_clean) == len(clean), (
        f"PRM returned {len(s_clean)} scores for {len(clean)} steps; the separator "
        f"convention for this checkpoint is not what this cell assumes")

    print("step scores")
    for i, (a, b) in enumerate(zip(s_clean, s_dirty)):
        mark = "  <- corrupted" if i == 2 else ""
        print(f"  step {i+1}   clean {a:.3f}   corrupted {b:.3f}{mark}")

    separates = s_dirty[2] < s_clean[2]
    print(f"\nPRM marks the corrupted step lower: {separates}")
    if not separates:
        PRM_READY = False
        print("It does not separate a known-false step from a true one, so its scores are")
        print("not usable as labels. Falling back to the checker alone.")


In [ ]:
PRM_THRESHOLD = 0.5          # below this, the PRM is calling the step unsound

if PRM_READY:
    for r in tqdm(records, desc="PRM scoring", unit="trace"):
        try:
            sc = prm_scores(r["question"], [s["text"] for s in r["steps"]])
        except Exception:
            sc = []
        if len(sc) != len(r["steps"]):
            r["prm_ok"] = False
            continue
        r["prm_ok"] = True
        for s, v in zip(r["steps"], sc):
            s["prm"] = float(v)
            s["prm_label"] = "false" if v < PRM_THRESHOLD else "true"
        r["prm_first_false"] = next(
            (i for i, s in enumerate(r["steps"]) if s["prm_label"] == "false"), None)
        r["prm_n_false"] = sum(s["prm_label"] == "false" for s in r["steps"])

    scored = [r for r in records if r.get("prm_ok")]
    print(f"\nPRM scored {len(scored)}/{len(records)} traces")
else:
    for r in records:
        r["prm_ok"] = False
    print("no PRM scores; every measurement below uses the checker alone")


In [ ]:
# Reliability: on steps the checker judged, how often does the PRM agree?
RELIABILITY = None
if PRM_READY:
    pairs = [(s["check"], s["prm_label"]) for r in records if r.get("prm_ok")
             for s in r["steps"] if s["check"] in ("true", "false") and "prm_label" in s]
    if pairs:
        agree = mean([a == b for a, b in pairs])
        false_steps = [(a, b) for a, b in pairs if a == "false"]
        true_steps = [(a, b) for a, b in pairs if a == "true"]
        RELIABILITY = {
            "n_steps_compared": len(pairs),
            "agreement": agree,
            "recall_on_false": mean([b == "false" for _, b in false_steps]) if false_steps else None,
            "specificity_on_true": mean([b == "true" for _, b in true_steps]) if true_steps else None,
            "n_false_steps": len(false_steps),
            "threshold": PRM_THRESHOLD,
        }
        print(f"compared on {len(pairs)} steps the checker could judge")
        print(f"  overall agreement ............. {agree:.1%}")
        if false_steps:
            print(f"  caught the arithmetically false {RELIABILITY['recall_on_false']:.1%} "
                  f"of {len(false_steps)}")
        if true_steps:
            print(f"  left the true ones alone ...... {RELIABILITY['specificity_on_true']:.1%} "
                  f"of {len(true_steps)}")
        print("\nWhere they disagree the checker is right, by construction. This number is what")
        print("the page must attach to any PRM-only claim about the steps the checker cannot see.")
    else:
        print("no overlapping steps to compare on; reliability is unmeasured and the page")
        print("may not report PRM-only judgements")


## 7. Detection: how often does outcome grading reward nonsense

Spec §3a. Among traces that reached the **right** answer, the share containing at least one step
the checker proved false. Outcome grading passes every one of them.

**This number may come out small, and the spec says so in advance.** An arithmetically false
step usually propagates into a wrong final answer, and the cases where it does not are often
method errors with valid arithmetic, which this checker cannot see. If it is small, the page
reports it as small and changes its verb. It does not get inflated.

Small still matters, and page 12 is why: STaR keeps the rationales that reach the right answer
and fine-tunes on them. This share is the contamination rate of that filter.


In [ ]:
right = [r for r in records if r["outcome_correct"]]
wrong = [r for r in records if not r["outcome_correct"]]

right_with_false = [r for r in right if r["n_false"] > 0]
DETECTION = {
    "n_right": len(right),
    "n_right_with_false_step": len(right_with_false),
    "base_rate": len(right_with_false) / max(len(right), 1),
    "n_wrong": len(wrong),
    "grader": "arithmetic checker",
}

print(f"{len(right)} traces reached the right answer.")
print(f"{len(right_with_false)} of them contain a step the checker proved false.")
print(f"\nbase rate: {DETECTION['base_rate']:.1%}")

if PRM_READY and RELIABILITY:
    pr = [r for r in right if r.get("prm_ok")]
    pr_false = [r for r in pr if r.get("prm_n_false", 0) > 0]
    DETECTION["prm_base_rate"] = len(pr_false) / max(len(pr), 1)
    DETECTION["prm_reliability"] = RELIABILITY["agreement"]
    print(f"the PRM, which also sees non-arithmetic steps, puts it at "
          f"{DETECTION['prm_base_rate']:.1%}")
    print(f"(attach its measured agreement of {RELIABILITY['agreement']:.0%} to that number)")

if DETECTION["base_rate"] == 0:
    print("\nZERO. The checker found nothing, which is a fact about the checker before it is")
    print("a fact about the model. See the spec's open question 4 before reporting this.")
elif DETECTION["base_rate"] < 0.05:
    print("\nSmall. Report it as small; the STaR framing is what gives it stakes.")


## 8. Localisation: how much of a wrong chain was fine

Spec §3b, and the measurement the page's Solution B has always taught without evidence.

Take traces that reached the **wrong** answer and where the checker finds a break at step *k* of
*n*. Two graders look at that same trace:

- **outcome** can only say *wrong*, which condemns all *n* steps;
- **process** points at step *k*, leaving steps 1 to *k-1* standing.

So the quantity is `k/n`, the share of the chain process grading exonerates. The second reading
tests something the page currently asserts rather than shows: whether a break really is one
step, or whether real chains carry several.


In [ ]:
localisable = [r for r in wrong if r["first_false"] is not None and r["n_steps"] > 1]

LOCALISATION = {"n_wrong": len(wrong), "n_localisable": len(localisable)}

if localisable:
    shares = sorted(r["first_false"] / r["n_steps"] for r in localisable)
    med = shares[len(shares) // 2]
    n_false_dist = Counter(r["n_false"] for r in localisable)
    LOCALISATION.update({
        "median_share_exonerated": med,
        "mean_share_exonerated": mean(shares),
        "share_with_exactly_one_false_step":
            n_false_dist[1] / max(len(localisable), 1),
        "n_false_step_distribution": dict(sorted(n_false_dist.items())),
    })

    print(f"{len(localisable)} of {len(wrong)} wrong traces have a findable break.\n")
    print(f"  outcome grading condemns .......... 100% of every one of those chains")
    print(f"  process grading condemns from ..... step {med:.0%} of the way in (median)")
    print(f"\n  -> the first {med:.0%} of the chain was fine, and outcome grading threw it away")
    print(f"\nbreaks per chain: {LOCALISATION['n_false_step_distribution']}")
    print(f"exactly one broken step: {LOCALISATION['share_with_exactly_one_false_step']:.0%} "
          f"of them")
    if LOCALISATION["share_with_exactly_one_false_step"] < 0.5:
        print("  ^ under half, so the page must not claim breaks localise to a single line.")
        print("    Follow the data: say how many, and keep Solution B as one case rather")
        print("    than as the rule.")
else:
    print(f"No wrong trace has a findable break ({len(wrong)} wrong traces).")
    print("Section 3b has nothing to plot. Either the checker's recall is too low on this")
    print("model's output, or the wrong answers fail for non-arithmetic reasons.")


In [ ]:
import matplotlib.pyplot as plt

if localisable:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.2),
                                   gridspec_kw={"width_ratios": [3, 2]})

    ax1.hist([s * 100 for s in shares], bins=10, range=(0, 100), color="#3b4a6b")
    ax1.axvline(med * 100, color="#c2703d", lw=2)
    ax1.set_xlabel("share of the chain before the break (%)")
    ax1.set_ylabel("wrong traces")
    ax1.set_title("where the break lands", fontsize=10, loc="left")
    ax1.spines[["top", "right"]].set_visible(False)

    ax2.barh([1, 0], [100, med * 100], height=0.5, color=["#8c8c8c", "#3b4a6b"])
    ax2.set_yticks([1, 0])
    ax2.set_yticklabels(["outcome", "process"])
    ax2.set_xlim(0, 100)
    ax2.set_xlabel("% of the chain condemned")
    ax2.set_title("what each grader throws away", fontsize=10, loc="left")
    ax2.spines[["top", "right"]].set_visible(False)

    fig.suptitle(f"{CFG['gen_model']} · greedy · MATH-500 L4-5 · n={len(localisable)} "
                 f"wrong traces with a findable break", fontsize=9, x=0.01, ha="left")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "figure_localisation.png", dpi=160, bbox_inches="tight")
    plt.show()
else:
    print("nothing to plot")


## 9. Selection: a process-aware rung on page 8's ladder

Spec §3c. Page 8 ranks five selectors that all read the final answer; this adds one that reads
the working. Among a problem's candidates, drop any containing a checker-false step, then apply
majority vote to what survives.

**Optional and self-skipping.** It needs page 8's dump, which needs page 7's sweep. If neither
has run, this section reports that and the page ships sections 7 and 8 alone. Nothing else here
depends on it.

Expect a small gain, and report it whatever it is. A null result is a real finding about how
much of the failure is method rather than arithmetic.


In [ ]:
PAGE8_TRACES = Path("out/page_07/traces.jsonl")     # page 8 selects over page 7's candidates
SELECTION = None

if not PAGE8_TRACES.exists():
    print(f"{PAGE8_TRACES} not found -- skipping the selector rung.")
    print("Run page 7's notebook first if you want this comparison; sections 7 and 8 above")
    print("are unaffected and are what the page leads with.")
else:
    from collections import defaultdict
    cands = defaultdict(list)
    for line in PAGE8_TRACES.open(encoding="utf-8"):
        if line.strip():
            r = json.loads(line)
            cands[r["problem_id"]].append(r)

    def majority(rs):
        if not rs:
            return None
        tally = Counter(r["extracted_answer"] for r in rs if r["extracted_answer"] is not None)
        return tally.most_common(1)[0][0] if tally else None

    plain, filtered, n_filtered_empty = [], [], 0
    for pid, rs in cands.items():
        truth = rs[0]["ground_truth"]
        keep = []
        for r in rs:
            steps = segment_steps(r["trace"], r["tokens"])
            if not any(check_step(s["text"])[0] == "false" for s in steps):
                keep.append(r)
        if not keep:
            n_filtered_empty += 1
            keep = rs                       # dropping everything is not a selection
        plain.append(bool(grade_answer(majority(rs), truth)))
        filtered.append(bool(grade_answer(majority(keep), truth)))

    SELECTION = {
        "n_problems": len(cands),
        "majority": mean(plain),
        "majority_after_step_filter": mean(filtered),
        "delta_pts": (mean(filtered) - mean(plain)) * 100,
        "n_problems_all_candidates_filtered": n_filtered_empty,
    }
    print(f"over {len(cands)} problems from page 7's dump")
    print(f"  majority vote ..................... {SELECTION['majority']:.1%}")
    print(f"  majority after dropping bad steps . {SELECTION['majority_after_step_filter']:.1%}")
    print(f"  delta ............................. {SELECTION['delta_pts']:+.1f} points")
    if abs(SELECTION["delta_pts"]) < 1:
        print("\nEssentially null. Say so: with this checker's recall, reading the working")
        print("buys almost nothing at selection time, and detection and localisation are")
        print("where step grading earns its keep.")


## 10. The three specimens

The page's interactive keeps its structure: the reader marks each step, then the labels are
revealed. What changes is that the traces are real and the labels are the checker's.

Three roles, the same three the current page uses:

| role | what it shows |
|---|---|
| `lucky` | right answer, at least one false step — outcome grading rewards it |
| `broken` | wrong answer, one findable break — outcome says *wrong*, process says *step k* |
| `clean` | right answer, nothing false — the graders agree |

Curated by filter, shortest first. Any role that cannot be filled is reported rather than
faked, and a missing role blocks translation for that panel.


In [ ]:
def pick(pool):
    ranked = sorted(pool, key=lambda r: (r["n_steps"], r["problem_id"]))
    ranked = [r for r in ranked if 2 <= r["n_steps"] <= 8] or ranked
    return ranked[0] if ranked else None


SPECIMENS, missing = {}, []
wanted = {
    "lucky":  [r for r in right if r["n_false"] > 0],
    "broken": [r for r in wrong if r["n_false"] == 1 and r["n_steps"] > 1],
    "clean":  [r for r in right if r["n_false"] == 0 and r["n_checkable"] > 0],
}
for role, pool in wanted.items():
    got = pick(pool)
    if got is None:
        missing.append(role)
        continue
    SPECIMENS[role] = {
        "role": role,
        "problem_id": got["problem_id"],
        "level": got["level"],
        "question": got["question"],
        "ground_truth": got["ground_truth"],
        "extracted_answer": got["extracted_answer"],
        "outcome_correct": got["outcome_correct"],
        "trace": got["trace"],
        "tokens": got["tokens"],
        "steps": got["steps"],
        "first_false": got["first_false"],
        "n_false": got["n_false"],
        "prm_scored": got.get("prm_ok", False),
    }
    marks = " ".join({"true": "+", "false": "X", "unknown": "?"}[s["check"]]
                     for s in got["steps"])
    print(f"{role:<7} {got['problem_id']:<30} {got['n_steps']} steps   {marks}")

print("\n  + checked true   X checked false   ? no numeric claim to check")

if missing:
    print(f"\nMISSING ROLES: {missing}. derived.json records this. A role with no specimen")
    print("cannot be translated to the page -- do not substitute an authored trace for it.")

if "broken" in SPECIMENS:
    sp = SPECIMENS["broken"]
    print(f"\n--- {sp['problem_id']}, the localisation case " + "-" * 30)
    for i, s in enumerate(sp["steps"]):
        flag = {"true": "  ", "false": "->", "unknown": "  "}[s["check"]]
        print(f"{flag} {i+1}. [{s['check']:<7}] {s['text'][:88]}")
    print(f"\noutcome grader: wrong (answered {sp['extracted_answer']}, "
          f"truth {sp['ground_truth']})")
    print(f"process grader: step {sp['first_false'] + 1} of {len(sp['steps'])}; "
          f"the {sp['first_false']} before it stand")


## 11. Artifacts

`derived.json` is the contract with the page: it reads that and nothing else, and no inference
runs at demonstration time. `traces.jsonl` carries every trace with its per-step labels and is
published as a download.

Note what the manifest has to record here that other pages do not: **two models with two
different jobs**, and the PRM's measured reliability. A reader who wants to dispute a step label
needs to know which grader produced it and how well that grader did on the steps we could check
independently.


In [ ]:
def git_sha():
    try:
        return subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True,
                              text=True, check=True).stdout.strip()
    except Exception:
        return None


def repo_revision(model_id):
    try:
        from huggingface_hub import model_info
        return model_info(model_id).sha
    except Exception:
        return None


manifest = {
    "run_id": RUN_ID,
    "notebook": "page_09_outcome_vs_process.ipynb",
    "page": 9,
    "run_mode": RUN_MODE,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "notebook_git_sha": git_sha(),
    "generator": {"id": CFG["gen_model"], "revision": repo_revision(CFG["gen_model"]),
                  "params": N_PARAMS, "dtype": str(DTYPE),
                  "prompt_suffix": COT_SUFFIX},
    "process_graders": {
        "arithmetic_checker": {
            "rule": "closed-form numeric on both sides, no symbols; labels true/false/unknown",
            "known_limit": "cannot see method errors with valid arithmetic",
        },
        "prm": ({"id": PRM_MODEL, "revision": repo_revision(PRM_MODEL),
                 "threshold": PRM_THRESHOLD, "reliability_vs_checker": RELIABILITY}
                if PRM_READY else None),
    },
    "decoding": {"strategy": "greedy", "do_sample": False, "max_new_tokens": MAX_NEW_TOKENS},
    "items": {"source": "MATH-500 (ch03 copy)", "filter": "level in {4, 5}",
              "n": len(items), "item_set_sha256": ITEM_SET_HASH, "seed": SEED},
    "grading": {"outcome": "reasoning_from_scratch.ch03.grade_answer"},
    "prompt_template": render_prompt("{question}") + COT_SUFFIX,
    "versions": {"torch": torch.__version__,
                 "transformers": __import__("transformers").__version__,
                 "device": str(device)},
    "integrity": {"truncation_rate": trunc / len(records),
                  "traces_generated_fresh": True,
                  "why_not_reused": "page 6's traces are truncated and forced, so their "
                                    "breaks are our intervention rather than the model's error"},
}

derived = {
    "manifest": manifest,
    "detection": DETECTION,
    "localisation": LOCALISATION,
    "selection": SELECTION,
    "prm_reliability": RELIABILITY,
    "specimens": SPECIMENS,
    "missing_roles": missing,
    "step_label_counts": dict(lab),
}

DERIVED_PATH.write_text(json.dumps(derived, indent=2, ensure_ascii=False), encoding="utf-8")
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

# traces.jsonl gets rewritten with the per-step labels attached
with TRACES_PATH.open("w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

kb = lambda p: p.stat().st_size / 1024
print(f"derived.json   {kb(DERIVED_PATH):>9.1f} KB   <- bundled with the page")
print(f"manifest.json  {kb(MANIFEST_PATH):>9.1f} KB   <- bundled with the page")
print(f"traces.jsonl   {kb(TRACES_PATH):>9.1f} KB   <- published as a download")


## Notes for re-derivation

- The checker is **high precision, low recall by design**. Every `false` label is verifiable by
  hand; every `unknown` is a step it declined to judge rather than one it approved.
- Detection and localisation are computed from the checker alone, so they stand whether or not
  the PRM ran. Any PRM-only claim must carry the reliability from section 6.
- Greedy decoding still diverges across hardware on a few items through floating-point
  differences. Expect the shape to reproduce, not every digit.
- These traces are **not** page 6's or page 7's, deliberately. See section 3.

What the page may and may not claim from this run is
[`docs/page_9_refactor.md`](../docs/page_9_refactor.md), which the translation step checks
itself against ([`docs/page_6_refactor.md`](../docs/page_6_refactor.md) §6).
